# Notebook 05: ML Models - Regression (AQI Prediction)

## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Train and evaluate multiple regression models to predict continuous AQI values. Compare model performance using RMSE, MAE, and R-squared metrics.

### Prerequisites:
- Complete Notebook 04 (generates X_train, X_test, y_train_reg, y_test_reg)
- Libraries: pandas, numpy, scikit-learn, xgboost, lightgbm

### Run Time: 20-30 minutes

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported!')

### Explanation:

- **from sklearn.linear_model import LinearRegression**: Simple linear regression (fits straight line to data).

- **from sklearn.linear_model import Ridge**: Linear regression with L2 regularization (prevents overfitting by penalizing large coefficients).

- **from sklearn.linear_model import Lasso**: Linear regression with L1 regularization (can zero out unimportant features).

- **from sklearn.ensemble import RandomForestRegressor**: Ensemble of decision trees (averages predictions from multiple trees).

- **from sklearn.ensemble import GradientBoostingRegressor**: Sequential tree building where each tree corrects previous errors.

- **from sklearn.svm import SVR**: Support Vector Regression (finds optimal hyperplane for regression).

- **from xgboost import XGBRegressor**: Extreme Gradient Boosting (fast, powerful gradient boosting implementation).

- **from lightgbm import LGBMRegressor**: Light Gradient Boosting (Microsoft's efficient gradient boosting).

- **from sklearn.metrics import mean_squared_error**: RMSE measures average prediction error (lower is better).

- **from sklearn.metrics import mean_absolute_error**: MAE measures average absolute error (easier to interpret).

- **from sklearn.metrics import r2_score**: R-squared shows how much variance is explained (0-1, higher is better).

- **from sklearn.model_selection import cross_val_score**: K-fold cross-validation for robust performance estimation.

In [ ]:
X_train = pd.read_csv(os.path.join('..', 'datasets', 'X_train.csv'))
X_test = pd.read_csv(os.path.join('..', 'datasets', 'X_test.csv'))
y_train = pd.read_csv(os.path.join('..', 'datasets', 'y_train_reg.csv')).values.ravel()
y_test = pd.read_csv(os.path.join('..', 'datasets', 'y_test_reg.csv')).values.ravel()
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

## Step 2: Define Evaluation Function

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    print(f'{model_name}:')
    print(f'  RMSE: {rmse:.2f}')
    print(f'  MAE: {mae:.2f}')
    print(f'  R²: {r2:.4f}')
    print(f'  CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 'r2': r2, 'cv_r2': cv_scores.mean(), 'y_pred': y_pred}
print('Evaluation function defined!')

### Explanation:

- **model.fit()**: Trains the model on training data (learns patterns).

- **model.predict()**: Makes predictions on test data.

- **np.sqrt(mean_squared_error)**: Calculates RMSE (Root Mean Square Error).

- **mean_absolute_error**: Calculates MAE (Mean Absolute Error).

- **r2_score**: Calculates R-squared (proportion of variance explained).

- **cross_val_score(cv=5)**: 5-fold cross-validation (trains on 4 folds, tests on 1, repeats 5 times).

- **scoring='r2'**: Uses R-squared as evaluation metric for cross-validation.

## Step 3: Train Linear Regression

In [ ]:
results = []
lr_results = evaluate_model(LinearRegression(), X_train, y_train, X_test, y_test, 'Linear Regression')
results.append(lr_results)

### Explanation:

- **LinearRegression()**: Creates linear regression model instance.

- **Baseline model**: Simple model to compare against more complex models.

## Step 4: Train Ridge Regression

In [ ]:
ridge_results = evaluate_model(Ridge(alpha=1.0), X_train, y_train, X_test, y_test, 'Ridge Regression')
results.append(ridge_results)

### Explanation:

- **Ridge(alpha=1.0)**: Creates Ridge regression with regularization strength alpha=1.0.

- **alpha**: Controls regularization strength (higher = more regularization, prevents overfitting).

## Step 5: Train Lasso Regression

In [ ]:
lasso_results = evaluate_model(Lasso(alpha=0.01), X_train, y_train, X_test, y_test, 'Lasso Regression')
results.append(lasso_results)

### Explanation:

- **Lasso(alpha=0.01)**: Creates Lasso regression with alpha=0.01.

- **L1 regularization**: Can eliminate irrelevant features by setting their coefficients to zero.

## Step 6: Train Random Forest

In [ ]:
rf_results = evaluate_model(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), 
                            X_train, y_train, X_test, y_test, 'Random Forest')
results.append(rf_results)

### Explanation:

- **n_estimators=100**: Creates 100 decision trees in the forest.

- **random_state=42**: Ensures reproducible results.

- **n_jobs=-1**: Uses all CPU cores for parallel training (faster).

## Step 7: Train Gradient Boosting

In [ ]:
gb_results = evaluate_model(GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42), 
                            X_train, y_train, X_test, y_test, 'Gradient Boosting')
results.append(gb_results)

### Explanation:

- **n_estimators=100**: Builds 100 sequential trees.

- **learning_rate=0.1**: Controls contribution of each tree (lower = slower learning, better generalization).

## Step 8: Train XGBoost

In [ ]:
xgb_results = evaluate_model(XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42), 
                             X_train, y_train, X_test, y_test, 'XGBoost')
results.append(xgb_results)

### Explanation:

- **XGBRegressor**: Extreme Gradient Boosting (optimized implementation).

- **Faster than GradientBoosting**: Uses parallel processing and regularization.

## Step 9: Train LightGBM

In [ ]:
lgbm_results = evaluate_model(LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42), 
                              X_train, y_train, X_test, y_test, 'LightGBM')
results.append(lgbm_results)

### Explanation:

- **LGBMRegressor**: Light Gradient Boosting Machine (Microsoft).

- **Leaf-wise growth**: More efficient than level-wise growth (faster training).

## Step 10: Compare All Models

In [ ]:
results_df = pd.DataFrame(results)
print('\n=== MODEL COMPARISON ===')
print(results_df[['model', 'rmse', 'mae', 'r2', 'cv_r2']].to_string(index=False))
best_model = results_df.loc[results_df['r2'].idxmax()]
print(f'\nBest Model: {best_model["model"]} (R² = {best_model["r2"]:.4f})')

## Step 11: Visualize Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = results_df['model']
axes[0].barh(models, results_df['rmse'], color='salmon')
axes[0].set_xlabel('RMSE (lower is better)')
axes[0].set_title('RMSE Comparison')
axes[1].barh(models, results_df['mae'], color='lightskyblue')
axes[1].set_xlabel('MAE (lower is better)')
axes[1].set_title('MAE Comparison')
axes[2].barh(models, results_df['r2'], color='lightgreen')
axes[2].set_xlabel('R² (higher is better)')
axes[2].set_title('R² Comparison')
plt.tight_layout()
plt.show()

## Step 12: Actual vs Predicted Plot (Best Model)

In [ ]:
best_result = results[results_df['r2'].idxmax()]
plt.figure(figsize=(10, 6))
plt.scatter(y_test, best_result['y_pred'], alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual AQI', fontsize=12, fontweight='bold')
plt.ylabel('Predicted AQI', fontsize=12, fontweight='bold')
plt.title(f'Actual vs Predicted - {best_result["model"]}', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Explanation:

- **plt.scatter()**: Scatter plot showing relationship between actual and predicted values.

- **Perfect prediction line**: Red dashed line shows ideal predictions (actual = predicted).

- **Points close to line**: Better model performance.

## Step 13: Save Best Model

In [ ]:
import joblib
best_model_obj = None
for r in results:
    if r['model'] == best_result['model']:
        best_model_obj = r['model']
        break
joblib.dump(best_model_obj, os.path.join('..', 'models', 'best_regression_model.pkl'))
results_df.to_csv(os.path.join('..', 'outputs', 'regression_results.csv'), index=False)
print(f'Best model ({best_result["model"]}) saved to models/ folder!')
print('READY FOR NOTEBOOK 06 (Classification Models)')

## Summary

Trained 7 regression models:
1. Linear Regression (baseline)
2. Ridge Regression (L2 regularization)
3. Lasso Regression (L1 regularization)
4. Random Forest (ensemble of trees)
5. Gradient Boosting (sequential trees)
6. XGBoost (optimized gradient boosting)
7. LightGBM (leaf-wise growth)

**Metrics:**
- RMSE: Root Mean Square Error
- MAE: Mean Absolute Error
- R²: R-squared (variance explained)
- CV R²: Cross-validated R²

**Next**: Notebook 06 - ML Classification Models (AQI Bucket Prediction)